# `StructurePreprocessor.encode()`

`encode` is the single feature/backend router: you pass one mixed `features` list and it dispatches each key to its owning encoder (`encode_dssp` / `encode_pdb` / `encode_pae` / `encode_domains`) via the feature registry, then merges the per-source tensors into one `[0, 1]`-normalized `dict_num` whose D-axis follows the requested feature order. You no longer need to know which source a key comes from or call `combine_dict_nums` yourself. Here we mix PDB ATOM features (`bfactor`, `plddt`) with an AlphaFold PAE summary (`pae_row_mean`) using the bundled `AF_TINY` fixture.

In [1]:
import warnings
import tempfile, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import aaanalysis as aa
import aaanalysis.utils as ut
aa.options['verbose'] = False
warnings.filterwarnings('ignore')

PDB_FIXTURES = Path(aa.__file__).resolve().parent / '_data' / 'pdb_test'
strp = aa.StructurePreprocessor(verbose=False)
df_seq = pd.DataFrame({'entry': ['AF_TINY'],
                       'sequence': ['ACDEFGHIKLMNPQRSTVWYACDEFGHIKL']})
# PAE sidecar under the <entry>.json name the resolver expects.
pae_dir = tempfile.mkdtemp()
shutil.copy(PDB_FIXTURES / 'AF_TINY_pae.json', Path(pae_dir) / 'AF_TINY.json')

# One call, keys from two different sources, interleaved in any order.
feats = ['bfactor', 'pae_row_mean', 'plddt']
dict_num, df_out = strp.encode(df_seq=df_seq, features=feats,
                              pdb_folder=str(PDB_FIXTURES), pae_folder=pae_dir,
                              return_df=True)
arr = dict_num['AF_TINY']
print('merged shape (L, D):', arr.shape, '-> D follows feats order', feats)

merged shape (L, D): (30, 3) -> D follows feats order ['bfactor', 'pae_row_mean', 'plddt']


In [2]:
# return_df=True echoes df_seq (kept entries) plus a combined 'encode_ok'
# column = AND of every routed backend's entry-level status.
aa.display_df(df_out, n_rows=10, show_shape=True)

DataFrame shape: (1, 3)


,entry,sequence,encode_ok
1,AF_TINY,ACDEFGHIKLMNPQRSTVWYACDEFGHIKL,True


**Per-feature failure isolation.** If one feature is unavailable (for example `depth`, which needs the external `msms` binary) or its encoder raises for an entry, only that feature's column(s) are filled with `NaN` while every other feature is kept and the entry is retained; a single `UserWarning` names the isolated feature. This replaces the old behavior where one unavailable feature `NaN`-ed the whole entry. Below, `depth` is isolated (its column is all `NaN`) while `bfactor` and `pae_row_mean` come through intact.

In [3]:
feats2 = ['bfactor', 'depth', 'pae_row_mean']
dn = strp.encode(df_seq=df_seq, features=feats2,
                pdb_folder=str(PDB_FIXTURES), pae_folder=pae_dir)
v = dn['AF_TINY']
for j, key in enumerate(feats2):
    print(f'{key:>14}: all-NaN column? {bool(np.isnan(v[:, j]).all())}')

       bfactor: all-NaN column? False
         depth: all-NaN column? True
  pae_row_mean: all-NaN column? False


**Further parameters.** `StructurePreprocessor.encode` also accepts: `domain_folder` — directory of per-entry chopping files, required when a domain key is requested unless `df_seq` already carries a `chopping` column; `ss_mode` / `gap_handling` — forwarded to `encode_dssp` when a DSSP key is requested; `plddt_disorder_threshold` — forwarded to `encode_pdb` for the `plddt_disorder` key; `local_window` / `pae_band_edges` — forwarded to `encode_pae`; `on_failure` — `'nan'` isolates failures (default), `'drop'` keeps only entries present in every routed source, `'raise'` re-raises; `return_df` — also return the `(dict_num, df_seq_out)` pair.

## Further parameters

`encode()` accepts every backend's option in one call; the router applies each only to the features that need it.

In [ ]:
# Every routing option, passed explicitly (unused ones are ignored per feature).
dict_num2 = strp.encode(
    df_seq=df_seq,
    features=['bfactor', 'plddt', 'pae_row_mean'],
    pdb_folder=str(PDB_FIXTURES),
    pae_folder=pae_dir,
    domain_folder=None,             # only used for domain-derived features
    ss_mode='ss3',                  # DSSP secondary-structure alphabet: 'ss3' | 'ss8'
    gap_handling='pad',             # missing residues: 'pad' (NaN) | 'omit'
    plddt_disorder_threshold=70.0,  # AF pLDDT cutoff for the disorder feature
    local_window=5,                 # residue window for local-window features
    pae_band_edges=(5, 15),         # PAE distance-band edges (near / mid / far)
    on_failure='nan',               # per-feature failure policy: 'nan' | 'drop' | 'raise'
)
print('all routing options applied:', dict_num2['AF_TINY'].shape)